# Confident and Wrong

**Topic:** Calibration &nbsp;·&nbsp; **Difficulty:** Medium

A model that says "90% sure" should be right 90% of the time. Most trained
classifiers are not: they are systematically overconfident, and the gap is
invisible in AUC or any other ranking metric, all of which only care about
the *order* of the scores, not their values.

Measure it with expected calibration error, then fix it with Platt scaling and
measure again.

## The data (created in the next cell)

- `p_val` `(4000,)`, `y_val` `(4000,)`: validation scores and labels for fitting
  the calibrator.
- `p_test` `(4000,)`, `y_test` `(4000,)`: for measuring.
- The scores are deliberately overconfident.

## Your task

With **sklearn and NumPy**:

1. Implement ECE with 15 equal-width bins over `[0, 1]`:
   `ECE = sum_b (n_b / n) * |accuracy_b - confidence_b|`, where confidence is the
   mean predicted probability in the bin and accuracy the fraction of positives.
   Skip empty bins.
2. `ece_raw`: ECE of `p_test`.
3. Fit Platt scaling: `LogisticRegression` on the **logit** of `p_val` against
   `y_val`, then apply it to `p_test`.
4. `ece_platt`: ECE of the calibrated test probabilities.
5. Confirm ranking is unchanged: `auc_raw` and `auc_platt` using
   `sklearn.metrics.roc_auc_score`.
6. Set `answers` to a dict with `"ece_raw"`, `"ece_platt"`, `"auc_raw"`,
   `"auc_platt"`, `"mean_conf_raw"` (mean predicted probability on test).

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks calibration improved by more than 2x while AUC is essentially unchanged.

In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    z = r.normal(0, 1.4, n)
    y = (r.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    sharp = 1 / (1 + np.exp(-2.6 * z))        # too confident on purpose
    return np.clip(sharp, 1e-6, 1 - 1e-6), y

p_val, y_val = make(4000, 91)
p_test, y_test = make(4000, 92)
print(f"val {p_val.shape}, test {p_test.shape}, "
      f"mean predicted {p_test.mean():.3f}, actual rate {y_test.mean():.3f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("ece_raw", [], 0.1484158097337371, -0.02485044854890336, 1e-06),
    ("ece_platt", [], 0.02600952569515021, -0.0043549833486626575, 1e-06),
    ("auc_raw", [], 0.7895287028702871, -0.13219711864766742, 1e-06),
    ("auc_platt", [], 0.7895287028702871, -0.13219711864766742, 1e-06),
    ("mean_conf_raw", [], 0.5008232739905367, -0.08385685476987347, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "ece_platt" in answers:
    _better = answers["ece_platt"] * 2 < answers["ece_raw"]
    _auc = abs(answers["auc_platt"] - answers["auc_raw"]) < 1e-6
    print(f"  {'calibration improves 2x':<24} {'ok' if _better else 'wrong'}  "
          f"({answers['ece_raw']:.4f} -> {answers['ece_platt']:.4f})")
    print(f"  {'AUC unchanged':<24} {'ok' if _auc else 'wrong'}")
    _ok = _ok and _better and _auc

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Platt scaling fits a one-dimensional logistic model on the logits, so it can only
stretch or compress the score axis, a strictly monotone transformation. That is
exactly why AUC does not move: ranking metrics are invariant to monotone maps,
and calibration is precisely the information they discard. It is also why you can
calibrate after the fact without retraining anything, and why calibration must be
fitted on held-out data rather than the training set.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def ece(p, y, n_bins=15):
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1], right=False), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if not m.any():
            continue
        total += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(total)

def logit(p):
    return np.log(p / (1 - p))

cal = LogisticRegression().fit(logit(p_val).reshape(-1, 1), y_val)
p_cal = cal.predict_proba(logit(p_test).reshape(-1, 1))[:, 1]

answers = {
    "ece_raw": ece(p_test, y_test),
    "ece_platt": ece(p_cal, y_test),
    "auc_raw": float(roc_auc_score(y_test, p_test)),
    "auc_platt": float(roc_auc_score(y_test, p_cal)),
    "mean_conf_raw": float(p_test.mean()),
}
```